# Bands and density of states of graphene-based systems with `pythtb`

In the DFT part of the course we have seen that, in graphene, the electronic states close to the Fermi energy are mainly derived from the carbon $p_z$ orbitals.  
For this reason we can use a very simple tight-binding model, with only one $p_z$ orbital per carbon atom and only nearest-neighbor hopping, to describe the $\pi$ bands of graphene and related nanomaterials.

In this notebook we use the [`pythtb`](https://www.physics.rutgers.edu/pythtb/) library to compute:

- band structures
- densities of states (DOS)

for graphene, a graphene nanoribbon, and a small graphene nanoflake.

Please also consult the [pythtb documentation](https://www.physics.rutgers.edu/pythtb/).

This simple model is not meant to reproduce all electronic states of carbon materials. It is only meant to capture the low-energy $\pi$-electron physics close to the Fermi level.

## Model and helper functions

We include only $p_z$ orbitals and only nearest-neighbor hopping with

$$
t = -2.8\ \mathrm{eV}.
$$

The short code cell below imports the libraries and defines a small helper function to build a broadened DOS from the eigenvalues.

In [ ]:
from IPython import display
from pythtb import *  # tight-binding model class

import numpy as np
import matplotlib.pyplot as plt


def gaussian(x_arr, x, fwhm):
    sigma = fwhm / 2.355
    prefactor = 1.0 / (sigma * np.sqrt(2 * np.pi))
    return prefactor * np.exp(-0.5 * ((x_arr - x) / sigma) ** 2)


def calc_dos(evals, fwhm=0.1, emin=-8.0, emax=8.0, de=0.01):
    e_arr = np.arange(emin, emax, de)
    dos = np.zeros_like(e_arr)
    for e in np.ravel(evals):
        dos += gaussian(e_arr, e, fwhm)
    return e_arr, dos

# Graphene

We start from pristine graphene. The unit cell contains two carbon atoms, one for each sublattice.  
After building the model, we compute:

- the DOS from a dense uniform sampling of the Brillouin zone
- the band structure along the high-symmetry path $\Gamma$–M–K–$\Gamma$

The image below shows the unit cell convention used here.

In [ ]:
# Graphene lattice vectors and orbital positions
lat = [[1.0, 0.0], [-0.5, np.sqrt(3.0) / 2.0]]
orb = [[0.0, 0.0], [1.0 / 3.0, 2.0 / 3.0]]

# Build the graphene model
graphene = tb_model(2, 2, lat, orb)

t = -2.8

# Nearest-neighbor hoppings
graphene.set_hop(t, 0, 1, [0, 0])
graphene.set_hop(t, 1, 0, [0, 1])
graphene.set_hop(t, 1, 0, [1, 1])

In [ ]:
display.Image("./unit_cell.png")

## DOS of graphene

For the DOS we need a good sampling of the full Brillouin zone, so we use a dense uniform $k$-mesh.

In [ ]:
kmesh = graphene.k_uniform_mesh([200, 200])
evals_dos = graphene.solve_all(kmesh)

e_arr, dos = calc_dos(evals_dos)

## Band structure of graphene

For the band structure we instead sample a one-dimensional path connecting high-symmetry points.

In [ ]:
path = [[0.0, 0.0], [0.5, 0.0], [1.0 / 3.0, 1.0 / 3.0], [0.0, 0.0]]
label = (r"$\Gamma$", r"$M$", r"$K$", r"$\Gamma$")
nk = 121

k_vec, k_dist, k_node = graphene.k_path(path, nk, report=False)
evals_band = graphene.solve_all(k_vec)

In [ ]:
fig, axs = plt.subplots(1, 2, sharey=True, figsize=(9, 4))

# Bands
axs[0].set_xlim(k_node[0], k_node[-1])
axs[0].set_xticks(k_node)
axs[0].set_xticklabels(label)
for x in k_node:
    axs[0].axvline(x=x, linewidth=0.5, color="k")
for band in evals_band:
    axs[0].plot(k_dist, band, "k-")
axs[0].set_ylabel("Energy [eV]")
axs[0].set_title("Band structure")

# DOS
axs[1].fill_betweenx(e_arr, dos)
axs[1].set_title("DOS")
axs[1].set_xlabel("States [a.u.]")

plt.tight_layout()
plt.show()

# Graphene nanoribbon

We now construct a graphene nanoribbon starting from graphene, making a supercell, and then cutting a one-dimensional strip.

The images below show the ribbon construction and the resulting 7-AGNR.

In [ ]:
display.Image("./gnrs.png")

In [ ]:
# Start again from graphene
lat = [[1.0, 0.0], [-0.5, np.sqrt(3.0) / 2.0]]
orb = [[0.0, 0.0], [1.0 / 3.0, 2.0 / 3.0]]

graphene = tb_model(2, 2, lat, orb)
graphene.set_hop(t, 0, 1, [0, 0])
graphene.set_hop(t, 1, 0, [0, 1])
graphene.set_hop(t, 1, 0, [1, 1])

# Build a supercell and cut a ribbon
supercell = graphene.make_supercell([[4, 0], [1, 2]])
one_dim_model = supercell.cut_piece(1, 0, glue_edgs=False)

# Remove two edge orbitals to obtain the 7-AGNR
gnr_model = one_dim_model.remove_orb([14, 15])

fig, ax = gnr_model.visualize(0, 1)

In [ ]:
display.Image("./7AGNR.png", width=400)

For a one-dimensional ribbon, both the band structure and the DOS can be obtained from a one-dimensional $k$ sampling.

In [ ]:
kmesh = gnr_model.k_uniform_mesh([100])
evals = gnr_model.solve_all(kmesh)

e_arr, dos = calc_dos(evals, fwhm=0.2)
kvals = np.asarray(kmesh).ravel()

In [ ]:
fig, axs = plt.subplots(1, 2, sharey=True, figsize=(9, 4))

# Bands
for band in evals:
    axs[0].plot(kvals, band, "k-")
axs[0].set_xlim([0.0, 0.5])
axs[0].set_ylabel("Energy [eV]")
axs[0].set_xlabel("k")
axs[0].set_title("Nanoribbon bands")

# DOS
axs[1].fill_betweenx(e_arr, dos)
axs[1].set_title("DOS")
axs[1].set_xlabel("States [a.u.]")

plt.tight_layout()
plt.show()

# Graphene nanoflake

Finally, we cut a finite graphene nanoflake from a larger graphene supercell.

Since the system is finite, there is no $k$-space sampling anymore: we simply diagonalize the Hamiltonian and then broaden the discrete eigenvalues to obtain a DOS-like plot.

In [ ]:
lat = [[1.0, 0.0], [0.5, np.sqrt(3.0) / 2.0]]
orb = [[1.0 / 3.0, 1.0 / 3.0], [2.0 / 3.0, 2.0 / 3.0]]

graphene = tb_model(2, 2, lat, orb)
graphene.set_hop(t, 0, 1, [0, 0])
graphene.set_hop(t, 1, 0, [1, 0])
graphene.set_hop(t, 1, 0, [0, 1])

supercell = graphene.make_supercell([[1, 1], [-1, 2]], to_home=True)
gnr_model = supercell.cut_piece(3, 1, glue_edgs=False)
flake_model = gnr_model.cut_piece(3, 0, glue_edgs=False)

flake_model.visualize(0, 1)

In [ ]:
evals = flake_model.solve_all()
e_arr, dos = calc_dos(evals, fwhm=0.1)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.fill_between(e_arr, dos, lw=2.0)
ax.set_xlim([np.min(e_arr), np.max(e_arr)])
ax.set_ylim([0, np.max(dos) * 1.05])
ax.set_xlabel("Energy [eV]")
ax.set_ylabel("Density of states [a.u.]")
ax.set_title("Nanoflake DOS")

fig.tight_layout()
plt.show()

# fig.savefig("flake.pdf")
# fig.savefig("flake.png", dpi=300)

## What to observe

While reading the plots, try to connect the results to the dimensionality of the system:

- **Graphene** has the well-known Dirac-like crossing close to the Fermi level.
- **Nanoribbons** can open a gap depending on their width and edge structure.
- **Nanoflakes** have fully discrete energy levels because they are finite systems.

This is the strength of a minimal $p_z$ tight-binding model: with very little input, it already captures much of the low-energy electronic structure of graphene-based materials.